# German Credit Dataset — EDA Snippets
### Credit Risk ML Study Programme

These snippets were pulled together from study sessions covering probability basics and their practical application in EDA. Work through them in order — each section connects to a concept covered in the study guide.

---

## 1. Setup

In [ ]:
# ══════════════════════════════════════════════════════
# Environment Setup
# ══════════════════════════════════════════════════════

# ── Standard Library ──────────────────────────────────
import os
import sys
import warnings

# ── Data ──────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ── Machine Learning ──────────────────────────────────
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Jupyter ───────────────────────────────────────────
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display

# ── Custom Utilities ──────────────────────────────────
sys.path.append(os.path.join(os.path.abspath('..'), 'Core Resources'))
import python_style_util as psu

# ══════════════════════════════════════════════════════
# Display Settings
# ══════════════════════════════════════════════════════

%matplotlib inline

# Pandas
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.colheader_justify', 'left')

# NumPy
np.set_printoptions(precision=4, suppress=True, linewidth=120)

# Jupyter
InteractiveShell.ast_node_interactivity = 'all'

# Style
psu.set_style()

In [ ]:

# Pandas Display Options

pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places on floats
pd.set_option('display.max_columns', None)               # show all columns
pd.set_option('display.max_rows', 100)                   # show up to 100 rows
pd.set_option('display.width', None)                     # don't wrap wide dataframes
pd.set_option('display.colheader_justify', 'left')       # left-align column headers
pd.set_option('display.precision', 4)                    # decimal precision for describe()
pd.set_option('display.large_repr', 'truncate')          # truncate instead of summary for large dfs

# Numpy Display Options

pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places on floats
pd.set_option('display.max_columns', None)               # show all columns
pd.set_option('display.max_rows', 100)                   # show up to 100 rows
pd.set_option('display.width', None)                     # don't wrap wide dataframes
pd.set_option('display.colheader_justify', 'left')       # left-align column headers
pd.set_option('display.precision', 4)                    # decimal precision for describe()
pd.set_option('display.large_repr', 'truncate')          # truncate instead of summary for large dfs

# Matplotlib Display Options

%matplotlib inline                        # render charts inline in notebook
plt.rcParams['figure.dpi'] = 120          # sharper figures
plt.rcParams['savefig.dpi'] = 150         # higher res when saving
plt.rcParams['savefig.bbox'] = 'tight'    # no clipped labels when saving
plt.rcParams['savefig.facecolor'] = 'white'  # white background on saved figures

# Warnings Options

warnings.filterwarnings('ignore')          # suppress all warnings

# or more selectively:

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Jupyter Notebooks display


InteractiveShell.ast_node_interactivity = 'all'  # print every expression not just last one




**Dataset:** German Credit Dataset — UCI 1994  
**Author:** Chuch  
**Date:** 2026-06-08  
**Objective:** Playing around with Data Types + Missing Values concepts to study  
**Status:** In Progress

In [ ]:
columns = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings', 'employment', 'installment_rate', 'personal_status', 'other_debtors',
    'residence_since', 'property', 'age', 'other_installments', 'housing',
    'existing_credits', 'job', 'dependents', 'telephone', 'foreign_worker', 'target'
]


df = pd.read_csv('../German Credit Data/german.data', sep=' ', header=None, names=columns)
df.head()
# Or load from CSV if you have it locally:
# df = pd.read_csv('german_credit.csv')

print(df.shape)
df.head()

(1000, 21)


,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status,other_debtors,...,property,age,other_installments,housing,existing_credits,job,dependents,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


---
## 2. Conditional Probability — Default Rate by Feature
**Concept:** P(default | feature) — does knowing this feature change the probability of default?

This is the foundation of feature selection. Compare each result against the base rate (unconditional default rate). Strong features pull the probability meaningfully away from the base rate.

In [ ]:
# Conditional default rate for a categorical feature
# Replace 'missed_payment' with any categorical column in your dataset
df.groupby('job')['target'].mean()

job
A171    1.318182
A172    1.280000
A173    1.295238
A174    1.344595
Name: target, dtype: float64

In [ ]:
# Conditional default rate for a continuous feature — bin it first
# Replace 'credit_amount' with any continuous column
df['credit_amount_band'] = pd.cut(df['credit_amount'], bins=5)
df.groupby('credit_amount_band')['default'].mean()

**What to look for:**
- Large separation between groups → strong feature signal
- Monotonic relationship (consistently increasing or decreasing) → clean signal
- Rate similar to base rate across all groups → weak feature, may not add much

---
## 3. Poisson Check — Mean vs Variance
**Concept:** For a Poisson distribution, mean and variance should be approximately equal.

Use this on count variables (e.g. number of missed payments, number of credit enquiries). If variance >> mean, the Poisson assumption is breaking down — likely because events aren't truly independent.

In [ ]:
# Replace 'num_missed_payments' with your count variable
print('Mean:    ', df['num_missed_payments'].mean())
print('Variance:', df['num_missed_payments'].var())

# If variance >> mean, Poisson assumption is likely violated (overdispersion)

---
## 4. Distribution Shape — Histogram and Skewness
**Concept:** Understanding the shape of your data informs which distribution to fit and whether Normal assumptions hold.

In [ ]:
# Histogram for a continuous feature
# Replace 'credit_amount' with any continuous column
df['credit_amount'].hist(bins=30)
plt.title('Distribution of Credit Amount')
plt.xlabel('Credit Amount')
plt.ylabel('Frequency')
plt.show()

print(f'Skewness: {df["credit_amount"].skew():.3f}')
# Skewness > 1 or < -1 suggests the Normal assumption may not hold

---
## 5. Joint and Marginal Probability — Crosstab
**Concept:** Joint probability is P(A and B). Marginal probability is P(A) regardless of B — found in the margins of the table.

`normalize=True` converts counts to probabilities. `margins=True` adds marginal totals.

In [ ]:
# Joint and marginal probability table
# Replace 'missed_payment' with any categorical feature
pd.crosstab(
    df['missed_payment'],
    df['default'],
    margins=True,
    normalize=True
)

**Reading the table:**
- Inner cells = joint probabilities P(feature value AND default outcome)
- Row/column totals = marginal probabilities
- Divide any inner cell by its row marginal to get the conditional probability

---
## 6. Correlation Matrix
**Concept:** Correlation measures how features move together. Ranges from -1 to +1. Important for spotting multicollinearity before modelling.

In [ ]:
# Pearson correlation matrix — best for roughly normal, linear relationships
df.corr()

In [ ]:
# Spearman correlation — more robust for skewed data and outliers
# Recommended for credit data which is often skewed
df.corr(method='spearman')

In [ ]:
# Heatmap — visual version of the correlation matrix
plt.figure(figsize=(12, 8))
sns.heatmap(
    df.corr(method='spearman'),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0
)
plt.title('Spearman Correlation Matrix')
plt.tight_layout()
plt.show()

**What to look for:**
- Features highly correlated with `default` → strong candidates to include
- Features highly correlated with *each other* → multicollinearity risk in logistic regression
- General rule: feature-to-feature correlation above 0.7 or below -0.7 warrants investigation

---
## 7. Expected Value — Portfolio Level
**Concept:** Expected Loss = PD × LGD × EAD. This is expected value applied to credit risk.

In [ ]:
# Simple expected loss calculation
# Assumes you have PD estimates and loan amounts in your dataframe
# Replace column names as needed

LGD = 0.45  # Assumption — adjust based on your dataset or research

df['expected_loss'] = df['PD'] * LGD * df['credit_amount']

print(f'Total Expected Loss: ${df["expected_loss"].sum():,.2f}')
print(f'Average Expected Loss per Loan: ${df["expected_loss"].mean():,.2f}')

---
## 8. Variance and Standard Deviation — Portfolio Risk
**Concept:** Variance measures spread around the mean. In credit risk, higher variance means more uncertainty around expected losses.

In [ ]:
# Variance and standard deviation of a key feature
# Replace 'credit_amount' with any numeric column
print(f'Mean:               {df["credit_amount"].mean():,.2f}')
print(f'Variance:           {df["credit_amount"].var():,.2f}')
print(f'Standard Deviation: {df["credit_amount"].std():,.2f}')

In [ ]:
# Compare variance across default vs non-default groups
# Higher variance in the default group signals more spread in that population
df.groupby('default')['credit_amount'].agg(['mean', 'var', 'std'])

---
## 9. Law of Total Probability — Segment Default Rates
**Concept:** Overall portfolio default rate = weighted sum of segment default rates.

Useful for understanding how the mix of customers drives the overall rate.

In [ ]:
# Replace 'risk_segment' with any categorical segmentation variable
segment_stats = df.groupby('risk_segment').agg(
    count=('default', 'count'),
    default_rate=('default', 'mean')
)

segment_stats['portfolio_share'] = segment_stats['count'] / len(df)
segment_stats['weighted_default_rate'] = segment_stats['default_rate'] * segment_stats['portfolio_share']

print(segment_stats)
print(f'\nTotal Portfolio Default Rate (Law of Total Probability): {segment_stats["weighted_default_rate"].sum():.2%}')
print(f'Check against actual base rate: {df["default"].mean():.2%}')

---
## 10. Probability Calibration Check
**Concept:** A well-calibrated model's predicted probabilities match observed frequencies. Run this after you have model predictions.

Come back to this section once you've built your logistic regression model.

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

# y_true = actual default outcomes (0 or 1)
# y_pred = model predicted probabilities
# Replace with your actual arrays after modelling

fraction_of_positives, mean_predicted = calibration_curve(
    y_true, y_pred, n_bins=10
)

plt.figure(figsize=(8, 6))
plt.plot(mean_predicted, fraction_of_positives, 's-', label='Model')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives (Actual Default Rate)')
plt.title('Calibration Curve')
plt.legend()
plt.show()

In [ ]:
# Brier Score — lower is better, 0 is perfect
from sklearn.metrics import brier_score_loss
print(f'Brier Score: {brier_score_loss(y_true, y_pred):.4f}')